# UFC Fight Winner Prediction Pipeline (Corrected)

### Features:
1. **Robust Preprocessing**: Uses One-Hot Encoding to handle 'Gender', 'Stance', and other categorical features automatically.
2. **Dynamic Leakage Removal**: Automatically drops post-fight columns like 'Finish' or 'Round'.
3. **Model Comparison**: Trains XGBoost, Random Forest, Gradient Boosting, Logistic Regression, and a Deep Neural Network.
4. **Production Ready**: Can save the pipeline and predict on completely new, unseen CSV files.

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import copy

# Set device for PyTorch
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [7]:
class DeepUFCNet(nn.Module):
    def __init__(self, input_dim, hidden_layers=[128, 64]):
        super(DeepUFCNet, self).__init__()
        layers = []
        in_dim = input_dim
        
        for h_dim in hidden_layers:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.3))
            in_dim = h_dim
        
        layers.append(nn.Linear(in_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class UFCFightDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [8]:
class UFCFightPredictor:
    def __init__(self):
        self.models = {}
        self.scaler = StandardScaler()
        self.training_columns = None  # To ensure new data has same columns as training data
        self.numeric_cols = []
        
        self.ml_models_config = {
            'XGBoost': XGBClassifier(n_estimators=1000, learning_rate=0.01, max_depth=5, 
                                     subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42),
            'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
            'RandomForest': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
            'GradientBoosting': GradientBoostingClassifier(n_estimators=200, random_state=42)
        }

    def _preprocess_data(self, df, fit=False):
        df = df.copy()
        
        # 1. Target Engineering
        y = None
        if 'Winner' in df.columns and fit:
            # Map Winner to 1 (Blue) / 0 (Red)
            df['BlueWin'] = df['Winner'].apply(lambda x: 1 if x == 'Blue' else 0)
            y = df['BlueWin']

        # 2. Exclude Non-Features & Leakage
        # Specific columns to drop
        drop_cols = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'Date', 'Location', 'Country']
        # Keywords indicating post-fight stats (Data Leakage)
        leak_keywords = ['Finish', 'TotalFightTime', 'Referee', 'Round', 'WinsBy', 'decision', 'KO', 'Submission']
        
        # Filter columns
        cols_to_drop = [c for c in df.columns if c in drop_cols or any(k.lower() in c.lower() for k in leak_keywords)]
        # Keep 'BlueWins'/'RedWins' etc as they are historical
        # But 'WinsBy' usually refers to the result of the current match in some datasets, 
        # IF 'BlueWinsBy...' refers to historical stats, we should keep them.
        # In ufc-master.csv, 'BlueWinsByKO' etc. are historical totals. Let's strictly drop only result columns.
        # Re-refining leakage: only drop things that don't exist pre-fight.
        # For safety, let's trust the explicit 'Winner', 'Finish' columns are the main leakage.
        
        # Let's perform a safer drop: 
        final_drop = ['Winner', 'BlueWin', 'RedFighter', 'BlueFighter', 'Date', 'Location', 'Country', 
                      'Finish', 'FinishDetails', 'FinishRoundTime', 'TotalFightTime']
        
        X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

        # 3. Handle Boolean & Categorical Features
        # Convert bools to int
        bool_cols = X.select_dtypes(include=['bool']).columns
        for c in bool_cols:
            X[c] = X[c].astype(int)

        # One-Hot Encode Categorical (Object) columns
        # This handles Gender, Stance, WeightClass automatically
        X = pd.get_dummies(X, drop_first=True)

        # 4. Alignment with Training Columns
        if fit:
            self.training_columns = X.columns.tolist()
            # Identify numeric columns for scaling (those that are not dummy vars ideally, but scaling all is fine)
            # We'll scale everything for simplicity and consistency for the Neural Net
            self.numeric_cols = X.columns.tolist()
        else:
            # Align columns: Add missing columns as 0, remove extra columns
            X = X.reindex(columns=self.training_columns, fill_value=0)

        # 5. Handle NaNs (Fill with 0 - assuming missing stats mean 0/unrecorded)
        X = X.fillna(0)

        # 6. Scaling
        if fit:
            X[self.numeric_cols] = self.scaler.fit_transform(X[self.numeric_cols])
        else:
            X[self.numeric_cols] = self.scaler.transform(X[self.numeric_cols])

        return X, y

    def train(self, csv_path):
        print(f"Loading data from {csv_path}...")
        df = pd.read_csv(csv_path)
        
        print("Preprocessing data (Features & Targets)...")
        X, y = self._preprocess_data(df, fit=True)
        
        print(f"Training Data Shape: {X.shape}")
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
        self.results = []

        # --- Train ML Models ---
        for name, model in self.ml_models_config.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            self.models[name] = model
            self.results.append({'Model': name, 'Accuracy': acc})

        # --- Train Neural Network ---
        print("Training Neural Network...")
        train_dataset = UFCFightDataset(X_train.values, y_train)
        test_dataset = UFCFightDataset(X_test.values, y_test)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        
        nn_model = DeepUFCNet(input_dim=X_train.shape[1]).to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(nn_model.parameters(), lr=0.001)

        best_acc = 0
        for epoch in range(20):
            nn_model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = nn_model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
            
            # Val
            nn_model.eval()
            with torch.no_grad():
                val_preds = (nn_model(test_dataset.X) > 0.5).float()
                val_acc = accuracy_score(test_dataset.y.cpu(), val_preds.cpu())
                if val_acc > best_acc:
                    best_acc = val_acc
                    self.models['NeuralNetwork'] = copy.deepcopy(nn_model)
        
        self.results.append({'Model': 'NeuralNetwork', 'Accuracy': best_acc})
        print("Training Complete.")

    def evaluate_models(self):
        return pd.DataFrame(self.results).sort_values(by='Accuracy', ascending=False)

    def predict_new_data(self, new_csv_path):
        if not self.models:
            raise ValueError("Models not trained.")
            
        new_df = pd.read_csv(new_csv_path)
        X_new, _ = self._preprocess_data(new_df, fit=False)
        
        # Use best model
        best_model_name = max(self.results, key=lambda x: x['Accuracy'])['Model']
        print(f"Predicting with best model: {best_model_name}")
        
        model = self.models[best_model_name]
        
        if best_model_name == 'NeuralNetwork':
            model.eval()
            with torch.no_grad():
                preds_proba = model(torch.tensor(X_new.values, dtype=torch.float32).to(device)).cpu().numpy()
                preds = (preds_proba > 0.5).astype(int).flatten()
        else:
            preds = model.predict(X_new)
            preds_proba = model.predict_proba(X_new)[:, 1]

        results = new_df[['RedFighter', 'BlueFighter']].copy()
        results['PredictedWinner'] = ['Blue' if p == 1 else 'Red' for p in preds]
        results['BlueProbability'] = preds_proba
        return results

In [9]:
# 1. Initialize and Train
predictor = UFCFightPredictor()
predictor.train('ufc-master.csv')

# 2. Show Results
print("\nModel Performance:")
display(predictor.evaluate_models())

Loading data from ufc-master.csv...
Preprocessing data (Features & Targets)...
Training Data Shape: (6541, 104)
Training XGBoost...
Training LogisticRegression...
Training RandomForest...
Training GradientBoosting...
Training Neural Network...
Training Complete.

Model Performance:


,Model,Accuracy
4,NeuralNetwork,0.664629
0,XGBoost,0.662338
3,GradientBoosting,0.656226
2,RandomForest,0.650879
1,LogisticRegression,0.633308


In [10]:
# 3. Test on Unseen Data (Simulated)
# Create a dummy file with missing winner info
# sample = pd.read_csv('ufc-master.csv').sample(5)
# sample.drop(columns=['Winner']).to_csv('upcoming_fights.csv', index=False)

# Predict
predictions = predictor.predict_new_data('upcoming.csv')
display(predictions)

Predicting with best model: NeuralNetwork


,RedFighter,BlueFighter,PredictedWinner,BlueProbability
0,Colby Covington,Joaquin Buckley,Blue,0.822705
1,Cub Swanson,Billy Quarantillo,Blue,0.517529
2,Manel Kape,Bruno Silva,Red,0.244567
3,Vitor Petrino,Dustin Jacoby,Red,0.319996
4,Adrian Yanez,Daniel Marcos,Blue,0.705723
5,Navajo Stirling,Tuco Tokkos,Red,0.079061
6,Michael Johnson,Ottman Azaitar,Red,0.222886
7,Joel Alvarez,Drakkar Klose,Red,0.148921
8,Sean Woodson,Fernando Padilla,Red,0.309590
9,Miles Johns,Felipe Lima,Blue,0.631272
